# StandUp4AI IoU Evaluation

**Goal:** Implement IoU-thresholded evaluation like StandUp4AI (EMNLP 2025)

**Protocol:**
1. Predict laughter spans (contiguous regions)
2. Match predicted spans to ground truth spans with IoU >= 0.2
3. Calculate precision/recall/F1 at IoU >= 0.2

**StandUp4AI baseline:** F1=0.51 @ IoU=0.2

In [ ]:
# 1. Mount & Setup
from google.colab import drive
drive.mount('/content/drive')
import os
import numpy as np
import pandas as pd

# Find base
for base in ['/content/drive/MyDrive/standup4ai', '/content/drive/Shareddrives/standup4ai']:
    if os.path.exists(base):
        break
else:
    raise FileNotFoundError('standup4ai folder not found')

BASE = base
print(f'BASE: {BASE}')

In [ ]:
# 2. Load libraries
import librosa
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GroupKFold

print('Libraries loaded')

In [ ]:
# 3. Extract features WITH timestamps per segment
def extract_with_timestamps(audio_path, t0, t1, label):
    dur = t1 - t0
    if dur < 0.1:
        return None
    try:
        y, sr = librosa.load(audio_path, sr=22050, offset=t0, duration=min(dur, 10.0), mono=True)
    except:
        return None
    if len(y) < sr * 0.05:
        return None
    
    hop = 512
    feat = []
    
    # RMS (5)
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    feat.extend([np.mean(rms), np.std(rms), np.max(rms), np.min(rms), np.median(rms)])
    
    # ZCR (3)
    zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop)[0]
    feat.extend([np.mean(zcr), np.std(zcr), np.max(zcr)])
    
    # Spectral centroid (2)
    try:
        cent = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0]
        feat.extend([np.mean(cent), np.std(cent)])
    except:
        feat.extend([0, 0])
    
    # Spectral bandwidth (2)
    try:
        bw = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop)[0]
        feat.extend([np.mean(bw), np.std(bw)])
    except:
        feat.extend([0, 0])
    
    # Spectral rolloff (2)
    try:
        rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, hop_length=hop)[0]
        feat.extend([np.mean(rolloff), np.std(rolloff)])
    except:
        feat.extend([0, 0])
    
    # Spectral flatness (2)
    try:
        flatness = librosa.feature.spectral_flatness(y=y, hop_length=hop)[0]
        feat.extend([np.mean(flatness), np.std(flatness)])
    except:
        feat.extend([0, 0])
    
    # Duration (1)
    feat.append(len(y) / sr)
    
    # F0 if segment long enough (3)
    if dur >= 0.5:
        try:
            f0, voiced_flag, _ = librosa.pyin(y, fmin=80, fmax=500, sr=sr, hop_length=512)
            f0 = np.nan_to_num(f0, nan=0)
            voiced = np.nan_to_num(voiced_flag, nan=0)
            feat.extend([np.mean(f0), np.std(f0), np.mean(voiced)])
        except:
            feat.extend([0, 0, 0])
    else:
        feat.extend([0, 0, 0])
    
    return {
        'start': t0,
        'end': t1,
        'duration': dur,
        'label': 1 if str(label).strip() == 'risa' else 0,
        'features': np.array(feat, dtype=np.float32)
    }

In [ ]:
# 4. Extract from all videos
AUDIO_DIR = os.path.join(BASE, 'audio')
LABELS_DIR = os.path.join(BASE, 'labels')

audio_files = [f for f in os.listdir(AUDIO_DIR) if f.endswith('.m4a') or f.endswith('.mp3')]
label_files = [f for f in os.listdir(LABELS_DIR) if f.endswith('.csv')]

video_ids_audio = set(f.replace('.m4a','').replace('.mp3','') for f in audio_files)
video_ids_labels = set(f.replace('.csv','') for f in label_files)
overlap = sorted(video_ids_audio & video_ids_labels)
print(f'Videos with both: {len(overlap)}')

all_data = []
for i, vid in enumerate(overlap):
    audio_path = os.path.join(AUDIO_DIR, f'{vid}.m4a')
    label_path = os.path.join(LABELS_DIR, f'{vid}.csv')
    if not os.path.exists(audio_path) or not os.path.exists(label_path):
        continue
    try:
        df = pd.read_csv(label_path)
    except:
        continue
    for _, seg in df.iterrows():
        result = extract_with_timestamps(audio_path, float(seg['t0']), float(seg['t1']), seg['label'])
        if result:
            result['video_id'] = vid
            all_data.append(result)
    if (i+1) % 10 == 0:
        print(f'  {i+1}/{len(overlap)}')

print(f'Extracted: {len(all_data)} segments')
pos_count = sum(d['label'] for d in all_data)
print(f'Positive: {pos_count} ({100*pos_count/len(all_data):.1f}%)')

In [ ]:
# 5. Prepare data
X = np.array([d['features'] for d in all_data])
y = np.array([d['label'] for d in all_data])
vids = np.array([d['video_id'] for d in all_data])

print(f'X shape: {X.shape}, y: {y.shape}')

n_videos = len(set(vids))
n_splits = min(5, n_videos)
print(f'Using {n_splits}-fold GroupKFold on {n_videos} videos')

In [ ]:
# 6. Train models
gkf = GroupKFold(n_splits=n_splits)
models = []
scalers = []

for tr_idx, te_idx in gkf.split(X, y, vids):
    sc = StandardScaler()
    Xtr = sc.fit_transform(X[tr_idx])
    
    clf = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
    clf.fit(Xtr, y[tr_idx])
    
    models.append(clf)
    scalers.append(sc)

print(f'Trained {len(models)} models')

In [ ]:
# 7. IoU Evaluation Function
def span_iou(s1, s2):
    inter_start = max(s1[0], s2[0])
    inter_end = min(s1[1], s2[1])
    inter = max(0, inter_end - inter_start)
    union = max(s1[1], s2[1]) - min(s1[0], s2[0])
    return inter / union if union > 0 else 0

def compute_iou_eval(pred_spans, gt_spans, iou_threshold=0.2):
    if len(pred_spans) == 0:
        return 0, 0, 0
    if len(gt_spans) == 0:
        return 0, 0, 0
    
    matched_gt = set()
    matched_pred = set()
    
    for pi, pred in enumerate(pred_spans):
        best_iou = 0
        best_gi = -1
        for gi, gt in enumerate(gt_spans):
            if gi in matched_gt:
                continue
            iou_val = span_iou(pred, gt)
            if iou_val >= iou_threshold and iou_val > best_iou:
                best_iou = iou_val
                best_gi = gi
        if best_gi >= 0:
            matched_pred.add(pi)
            matched_gt.add(best_gi)
    
    tp = len(matched_pred)
    fp = len(pred_spans) - tp
    fn = len(gt_spans) - len(matched_gt)
    
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    
    return prec, rec, f1

# Test
p, r, f = compute_iou_eval([(0.0, 1.0), (2.0, 3.0)], [(0.5, 1.5)], iou_threshold=0.2)
print(f'IoU test: P={p:.2f} R={r:.2f} F1={f:.2f}')

In [ ]:
# 8. Evaluate with IoU at multiple thresholds
iou_thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
all_results = {th: [] for th in iou_thresholds}

for fold_idx, (tr_idx, te_idx) in enumerate(gkf.split(X, y, vids)):
    Xte = scalers[fold_idx].transform(X[te_idx])
    probs = models[fold_idx].predict_proba(Xte)[:, 1]
    
    te_data = [all_data[i] for i in te_idx]
    te_vids = vids[te_idx]
    
    for th in iou_thresholds:
        video_f1s = []
        
        for vid in set(te_vids):
            vid_mask = te_vids == vid
            vid_data = [te_data[i] for i in range(len(te_data)) if vid_mask[i]]
            vid_probs = probs[vid_mask]
            
            # Predicted spans (prob >= 0.5)
            pred_spans = [(d['start'], d['end']) for i, d in enumerate(vid_data) if vid_probs[i] >= 0.5]
            
            # GT spans
            gt_spans = [(d['start'], d['end']) for d in vid_data if d['label'] == 1]
            
            _, _, f1 = compute_iou_eval(pred_spans, gt_spans, iou_threshold=th)
            video_f1s.append(f1)
        
        all_results[th].append(np.mean(video_f1s))

print('\n=== IoU Evaluation Results ===')
print(f'{"IoU Threshold":<15} {"F1 Mean":<10} {"F1 Std":<10}')
print('-' * 35)
for th in iou_thresholds:
    mean_f1 = np.mean(all_results[th])
    std_f1 = np.std(all_results[th])
    print(f'IoU >= {th:<10.1f} {mean_f1:<10.4f} {std_f1:<10.4f}')

print(f'\nStandUp4AI baseline: F1=0.51 @ IoU=0.2')

In [ ]:
# 9. Vary prediction threshold
print('\n=== Varying Prediction Threshold (at IoU=0.2) ===')
pred_thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
best_f1 = 0
best_th = 0.5

for pth in pred_thresholds:
    all_video_f1s = []
    
    for fold_idx, (tr_idx, te_idx) in enumerate(gkf.split(X, y, vids)):
        Xte = scalers[fold_idx].transform(X[te_idx])
        probs = models[fold_idx].predict_proba(Xte)[:, 1]
        
        te_data = [all_data[i] for i in te_idx]
        te_vids = vids[te_idx]
        
        for vid in set(te_vids):
            vid_mask = te_vids == vid
            vid_data = [te_data[i] for i in range(len(te_data)) if vid_mask[i]]
            vid_probs = probs[vid_mask]
            
            pred_spans = [(d['start'], d['end']) for i, d in enumerate(vid_data) if vid_probs[i] >= pth]
            gt_spans = [(d['start'], d['end']) for d in vid_data if d['label'] == 1]
            
            _, _, f1 = compute_iou_eval(pred_spans, gt_spans, iou_threshold=0.2)
            all_video_f1s.append(f1)
    
    mean_f1 = np.mean(all_video_f1s)
    print(f'P >= {pth:<10.1f} -> IoU=0.2 F1 = {mean_f1:.4f}')
    if mean_f1 > best_f1:
        best_f1 = mean_f1
        best_th = pth

print(f'\nBest: pred_th={best_th}, IoU=0.2 F1={best_f1:.4f}')
print(f'Baseline: F1=0.51 @ IoU=0.2')

In [ ]:
# 10. Save results
results = {
    'n_samples': len(all_data),
    'n_videos': len(overlap),
    'iou_results': {th: float(np.mean(all_results[th])) for th in iou_thresholds},
    'best_pred_th': best_th,
    'best_iou_f1': float(best_f1),
    'baseline_f1': 0.51
}

import json
out_path = os.path.join(BASE, 'iou_results.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to: {out_path}')